# ReTReK route scoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laboratoire-de-Chemoinformatique/SynPlanner/blob/main/tutorials/19_Retrek_Route_Scoring.ipynb)

This tutorial scores finished synthesis routes with the ReTReK CD, AS, and RD reaction heuristics. Scoring happens **after** planning and consumes detached `Route` objects: it does not walk the MCTS tree or change the search score.

STScore is intentionally omitted because its matching formula is being corrected. MCTS-time ReTReK evaluation is separate future work.

In [ ]:
# Colab setup — this cell does nothing when you run the notebook locally.
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [
            "pip",
            "install",
            "-q",
            "git+https://github.com/Laboratoire-de-Chemoinformatique/SynPlanner.git@main",
        ],
        check=True,
    )
    print("SynPlanner installed. If an import below fails, restart the runtime.")


## The post-processing boundary

The data flow is `tree.routes()` → `list[Route]` → `RetrekRouteScorer.rank(routes)`. Each `Route` carries ordered `Step` objects, the exact product disconnected by each step, and the availability verdict for its leaves. That is all CDScore, ASScore, and RDScore need.

In [ ]:
from chython import smiles

from synplan.chem.reaction.reactor import Reaction
from synplan.chem.reaction.routes.quality.retrek import (
    RetrekRouteScorer,
    RetrekRouteScoringConfig,
)
from synplan.chem.reaction.routes.route import Route, RouteProvenance, Step
from synplan.chem.utils import molecule_key


## Build two small detached routes

Normally these objects come from `tree.routes()` or `read_routes_json(..., as_routes=True)`. Here they are built directly so the scoring example is fast and self-contained. Both routes use the same two precursors. The ring-forming route has two available leaves; the linear route records one unresolved leaf. Their deliberately different search scores show that ReTReK route quality is independent of MCTS provenance.

In [ ]:
def example_route(product_smiles, *, unresolved=(), search_score=None):
    reactants = [smiles("CCC"), smiles("CCO")]
    product = smiles(product_smiles)
    reaction = Reaction(reactants, [product])
    return Route(
        steps=(Step(reaction, product),),
        unresolved=frozenset(molecule_key(smiles(value)) for value in unresolved),
        provenance=RouteProvenance(search_score=search_score),
    )

ring_route = example_route("C1CCCCC1", search_score=0.10)
linear_route = example_route("CCCCCO", unresolved=("CCO",), search_score=0.95)
routes = [linear_route, ring_route]
routes


## Configure and score

The default configuration enables CDScore, ASScore, and RDScore with relative weights 5.0, 0.5, and 2.0. The step aggregate divides by the sum of available weights, so it remains in `[0, 1]`.

In [ ]:
config = RetrekRouteScoringConfig()
scorer = RetrekRouteScorer(config)

[
    {
        "target": str(route.target),
        "search_score": route.provenance.search_score,
        "step_scores": scorer.step_scores(route),
        "retrek_route_score": scorer.score(route),
    }
    for route in routes
]


The ring route receives the higher ReTReK score even though its stored search score is lower. `rank()` orders by the scorer's own number.

In [ ]:
ranked = scorer.rank(routes)
[(str(route.target), scorer.score(route)) for route in ranked]


## Use with planning results

After a normal search the complete workflow is simply:

```python
routes = tree.routes()
ranked_routes = RetrekRouteScorer().rank(routes)
```

The scorer reads `step.product`, `step.reaction.reactants`, and the route's leaf verdict. It never needs `tree.parents`, `tree.nodes`, or a building-block catalogue. Routes restored from JSON can be scored the same way.

## STScore and MCTS-time scoring

STScore is not enabled here. Its corrected definition and rule-matching tests must land before it is used. The route interface is ready for that work: an ST-enabled scorer will receive a resolver from each `Step` (including its `StepOrigin`) to the applicable canonical retro reactor.

ReTReK inside MCTS is also intentionally not implemented by this tutorial or scorer. That requires a separate design for the selection term and its interaction with back-propagated values.